In [0]:
%run ../00_common/data_utils

In [0]:
def generate_acsline_ukey(task_id):

    # 选择源系统(ACS和LineBind)
    t_merge_exclude_consumer_config = (
        spark.table(f"{get_env_config('config_database')}.t_merge_exclude_consumer_config")
        .filter(F.col("tmec_type").isin(SOURCE_TYPE_ACS, SOURCE_TYPE_LINEBIND))
        .select(
            F.col("tmec_marketcode").alias("exclude_mrkt"),
            F.col("tmec_sourcesystemcode").alias("exclude_srcs_code"),
        )
        .distinct()
    )

    # 最新ukey数据
    new_ukey_df = get_ukey_group_by_process(task_id) 


    #Batch Data 筛选 ACS&LineBind Consuemr 进行Uid计算
    t_clean_consumer = (
        spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer")
        .filter(F.col("task_id") == task_id)
        .filter(F.col("is_include") == True)
    )

    # 1. batch acsline profile
    acsline_clean_df = (
        t_clean_consumer
        .join(t_merge_exclude_consumer_config,
            (F.col("srcc_srcs_code") == F.col("exclude_srcs_code")) & (F.col("srcc_mrkt_code") == F.col("exclude_mrkt")),
            "inner"
        )
    )

    # 2.1 batch regular profile
    regular_clean_df = (
        t_clean_consumer.alias("tcc")
        .join(t_merge_exclude_consumer_config,
            (F.col("srcc_srcs_code") == F.col("exclude_srcs_code")) & (F.col("srcc_mrkt_code") == F.col("exclude_mrkt")),
            "left_anti"
        )
        .join(new_ukey_df.alias("nud"), (F.col("tcc.srcc_mrkt_code") == F.col("nud.mrkt_code")) & (F.col("tcc.srcc_id") == F.col("nud.srcc_id")), "left")
        .select(
            "tcc.srcc_mrkt_code",
            "tcc.srcc_brnd_code",
            "tcc.srcc_consumerid",
            "nud.new_consumermdmkey"
        )
    )
    

    # 2.2 master profile
    t_master_consumer = (spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer").alias("a")
        .select(
            F.col("a.scon_mrkt_code"),
            F.col("a.scon_brnd_code"),
            F.col("a.scon_srcs_code"),
            F.col("a.scon_consumerid"),
            F.col("a.consumermdmkey")
        )
    )

    # 3.1 generate acsline ukey
    acsline_ukey_df = (acsline_clean_df.alias("cc_tab")
        .join(
            t_master_consumer.alias("mc_tab1"),
            (F.col("cc_tab.srcc_mrkt_code")    == F.col("mc_tab1.scon_mrkt_code")) &
            (F.col("cc_tab.srcc_universalkey") == F.col("mc_tab1.consumermdmkey")) &
            (F.col("cc_tab.srcc_srcs_code")    != F.col("mc_tab1.scon_srcs_code")),
            "left" 
        )
        .join(
            t_master_consumer.alias("mc_tab2"),
            (F.col("cc_tab.srcc_mrkt_code")      == F.col("mc_tab2.scon_mrkt_code")) &
            (F.col("cc_tab.srcc_brnd_code")       == F.col("mc_tab2.scon_brnd_code")) &
            (F.col("cc_tab.SRCC_MASTERCONSUMERID") == F.col("mc_tab2.scon_consumerid")),
            "left"
        )
        .join(
            regular_clean_df.alias("rc_tab"),
            (F.col("cc_tab.srcc_mrkt_code")      == F.col("rc_tab.srcc_mrkt_code")) &
            (F.col("cc_tab.srcc_brnd_code")       == F.col("rc_tab.srcc_brnd_code")) &
            (F.col("cc_tab.SRCC_MASTERCONSUMERID") == F.col("rc_tab.srcc_consumerid")),
            "left"
        )
        .withColumn("new_consumermdmkey",
            F.when(
                F.coalesce(F.col("mc_tab1.consumermdmkey"), F.lit("")) != "",
                F.col("mc_tab1.consumermdmkey")
            ).when(
                F.coalesce(F.col("mc_tab2.consumermdmkey"), F.lit("")) != "",
                F.col("mc_tab2.consumermdmkey")
            ).when(
                F.coalesce(F.col("rc_tab.new_consumermdmkey"), F.lit("")) != "",
                F.col("rc_tab.new_consumermdmkey")
            ).otherwise(
                F.col("cc_tab.srcc_universalkey")
            )
        )
        .select(
            F.expr("uuid()").alias("matc_id"),
            F.col("cc_tab.SRCC_ID").alias("srcc_id"),
            F.col("cc_tab.SRCC_MRKT_CODE").alias("mrkt_code"),
            F.col("cc_tab.SRCC_BRND_CODE").alias("brnd_code"),
            F.col("cc_tab.SRCC_SRCS_CODE").alias("source_code"),
            F.col("cc_tab.SRCC_CONSUMERID").alias("consumer_id"),
            F.col("cc_tab.SRCC_SOURCETIMESTAMP").alias("source_timestamp"),
            F.col("new_consumermdmkey"),
            F.lit(None).alias("master_scon_id"),
            F.lit(None).alias("master_consumermdmkey"),
            F.lit(None).alias("master_recode_create_time"),
            F.lit(False).alias("is_master_recode"),
            F.lit(None).alias("gid"),
            F.lit(None).alias("grp_size"),
            F.col("cc_tab.batch_id"),
            F.lit(task_id).alias("task_id"),
            F.lit(MATCH_TYPE_ACSLINE_STR).alias("match_type"),
            F.current_timestamp().alias("creation_dt")
        )
        .withColumn("row_num", F.row_number().over(Window.partitionBy("mrkt_code", "SRCC_ID").orderBy(F.col("new_consumermdmkey").desc())))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    

    # 3.2 更新为最新ukey
    latest_ukey_df =  (new_ukey_df
        .filter(F.col("is_master_recode") == True)
        .filter(F.col("master_consumermdmkey") != F.col("new_consumermdmkey"))
        .select(
            F.col("mrkt_code"), 
            F.col("master_consumermdmkey").alias("old_consumermdmkey"), 
            F.col("new_consumermdmkey").alias("latest_consumermdmkey")
        )
        .withColumn("row_num", F.row_number().over(Window.partitionBy("mrkt_code", "old_consumermdmkey").orderBy(F.col("latest_consumermdmkey").desc())))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    latest_acsline_ukey_df = (acsline_ukey_df.alias("au")
        .join(latest_ukey_df.alias("lu"), 
              (F.col("au.mrkt_code") == F.col("lu.mrkt_code")) & (F.col("au.new_consumermdmkey") == F.col("lu.old_consumermdmkey")), 
              "left"
        )
        .select(
            F.col("au.*"),
            F.col("lu.latest_consumermdmkey")
        )
        .withColumn("new_consumermdmkey", F.coalesce(F.col("latest_consumermdmkey"), F.col("new_consumermdmkey")))
        .drop("latest_consumermdmkey")
    )


    save_to_target_table(
        latest_acsline_ukey_df,
        f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_ukey_group",
        f"task_id = '{task_id}' and match_type = '{MATCH_TYPE_ACSLINE_STR}' "
    )

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")


with StepLogger("4.2_ukey_match_acsline", "04-2", "consumerlist", task_id=task_id) as logger:
    generate_acsline_ukey(task_id)